# PyCoMo discretized growth #
This tutorial show-cases the use of discretized growth for optimization across the full solution space. 
With this, linearizing the model with fixed abundance or growth rate is no longer necessary. 
However, this approach ustilizes binary variable, using a MILP to solve. 
This is faster than non-linear solvers, but slower than LPs.

The implementation is based on an idea by Salvy and Hatzimanikatis:

Salvy, P., Hatzimanikatis, V. 
The ETFL formulation allows multi-omics integration in thermodynamics-compliant metabolism and expression models. 
Nat Commun 11, 30 (2020). 
https://doi.org/10.1038/s41467-019-13818-7

The expected runtime for this notebook is less than 10 minutes.

In [1]:
import cobra
import pycomo

2026-03-15 15:39:37,627 - PyCoMo - INFO - Logger initialized.


In [2]:
cobra.Configuration().solver="glpk"

In [3]:
pycomo.configure_logger(level="info")

## Base Model ##

In [4]:
def construct_single_org_choice(is_one=True):
    model_id = 1 if is_one else 2
    other_id = 2 if is_one else 1
    model = cobra.Model()

    model.name = f"Org{model_id}"
    
    model.add_metabolites([cobra.Metabolite(i) for i in [
        "S", 
        f"X{model_id}", 
        "Y1",
        "Y2",
        "A", 
        "P", 
        "BM"]])
    model.add_reactions([cobra.Reaction(i) for i in [
        f"EX_X{model_id}", 
        "EX_Y1", 
        "EX_Y2", 
        f"TP_X{model_id}", 
        "TP_Y1", 
        "TP_Y2", 
        "v4", 
        "v5", 
        "bio"]])

    for internal_met in [
        "S", 
        "A", 
        "P", 
        "BM"
    ]:
        model.metabolites.get_by_id(internal_met).compartment = "i"

    for external_met in [
        f"X{model_id}", 
        "Y1",
        "Y2",
    ]:
        model.metabolites.get_by_id(external_met).compartment = "e"

    for met in model.metabolites:
        met.name = met.id

    for rxn in model.reactions:
        rxn.name = rxn.id
    
    model.reactions.get_by_id(f"EX_X{model_id}").add_metabolites({f"X{model_id}": -1})
    model.reactions.get_by_id(f"TP_X{model_id}").add_metabolites({f"X{model_id}": -1, "S": 1})

    model.reactions.get_by_id(f"EX_Y{model_id}").add_metabolites({f"Y{model_id}": -1})
    model.reactions.get_by_id(f"EX_Y{other_id}").add_metabolites({f"Y{other_id}": -1})
    model.reactions.get_by_id(f"TP_Y{model_id}").add_metabolites({f"Y{model_id}": -1, "A": 1})
    model.reactions.get_by_id(f"TP_Y{other_id}").add_metabolites({f"Y{other_id}": -1, "P": 1})
    
    model.reactions.get_by_id("v4").add_metabolites({"S": -1, "BM": 1, "P": 1})
    model.reactions.get_by_id("v5").add_metabolites({"A": -1, "BM": 1})
    
    model.reactions.get_by_id("bio").add_metabolites({"BM": -1})

    model.reactions.get_by_id(f"EX_X{model_id}").bounds = (-1000.,0.)
    model.reactions.get_by_id(f"TP_X{model_id}").bounds = (0.,1.)

    model.reactions.get_by_id(f"EX_Y{model_id}").bounds = (0.,1000.)
    model.reactions.get_by_id(f"EX_Y{other_id}").bounds = (0.,1000.)
    model.reactions.get_by_id(f"TP_Y{model_id}").bounds = (0.,1.)
    model.reactions.get_by_id(f"TP_Y{other_id}").bounds = (-1.,0.)
    
    model.reactions.get_by_id("v4").bounds = (0.,1000.)
    model.reactions.get_by_id("v5").bounds = (0.,1000.)
    
    model.reactions.get_by_id("bio").bounds = (0.,1000.)

    model.objective = "bio"

    return model

In [5]:
def create_toy_pycomo_model():
    model1 = construct_single_org_choice(is_one=True)
    model2 = construct_single_org_choice(is_one=False)
    
    single_org1 = pycomo.SingleOrganismModel(model1, name="org1")
    single_org2 = pycomo.SingleOrganismModel(model2, name="org2")
    
    com_model = pycomo.CommunityModel([single_org1, single_org2], name="SymmetricToy")
    
    com_model.medium = {'EX_X1_medium': 1000.0,
     'EX_Y1_medium': 0.,
     'EX_Y2_medium': 0.,
     'EX_X2_medium': 1000.0}
    com_model.apply_medium()
    return com_model

## Find maximum growth rate ##
The maximum growth rate is 2.0 at 50%/50% abundance of the two organisms. It can be found quickly by the discretized growth rate:

In [6]:
com_model = create_toy_pycomo_model()

2026-03-15 15:39:37,924 - PyCoMo - INFO - No community model generated yet. Generating now:
2026-03-15 15:39:37,930 - PyCoMo - INFO - Identified biomass reaction from objective: bio
2026-03-15 15:39:37,930 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-03-15 15:39:37,931 - PyCoMo - INFO - Biomass reaction bio is a boundary reaction.
2026-03-15 15:39:37,931 - PyCoMo - INFO - Moving reactant BM out of the boundary compartment into compartment 'bio', to avoid further boundary reactions being added to it.
2026-03-15 15:39:37,956 - PyCoMo - INFO - Identified biomass reaction from objective: bio
2026-03-15 15:39:37,956 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-03-15 15:39:37,957 - PyCoMo - INFO - Biomass reaction bio is a boundary reaction.
2026-03-15 15:39:37,958 - PyCoMo - INFO - Moving reactant BM out of the boundary compartment into compartment 'bio', to avoid further boundary reactions being added

In [7]:
pycomo.helper.discretized_growth.relax_model_linearisation(com_model)
f_vars = pycomo.helper.discretized_growth.init_f_variables(com_model)
pycomo.helper.discretized_growth.discretize_growth_of_model(com_model, f_vars, mu_lb=0., mu_ub=100., n_mu_bins=1000)

[0 <= 1 <= 1, 0 <= 2 <= 1, 0 <= 4 <= 1, 0 <= 8 <= 1, 0 <= 16 <= 1, 0 <= 32 <= 1, 0 <= 64 <= 1, 0 <= 128 <= 1, 0 <= 256 <= 1, 0 <= 512 <= 1]


In [8]:
com_model.model.objective = "community_biomass"
sol = com_model.model.optimize("maximize")
sol

,fluxes,reduced_costs
org1_TF_X1_org1_e,-0.50,NaN
org1_TF_Y1_org1_e,-0.50,NaN
org1_TF_Y2_org1_e,0.50,NaN
org1_TP_X1,0.50,NaN
org1_TP_Y1,0.50,NaN
...,...,...
SK_org2_bio_ub,4.99,NaN
SK_org2_to_community_biomass_ub,4.99,NaN
f_final,1.00,NaN
abundance_reaction,0.00,NaN


## Find maximum of flux ##
flux v4 of organism 1 can reach its maximum flux of 1.0 at a growth rate 1.0 - not at maximum growth. See if the optimum is found:

In [9]:
com_model.model.objective = "org1_v4"
sol = com_model.model.optimize("maximize")
sol

,fluxes,reduced_costs
org1_TF_X1_org1_e,-1.0,NaN
org1_TF_Y1_org1_e,0.0,NaN
org1_TF_Y2_org1_e,1.0,NaN
org1_TP_X1,1.0,NaN
org1_TP_Y1,0.0,NaN
...,...,...
SK_org2_bio_ub,-0.0,NaN
SK_org2_to_community_biomass_ub,-0.0,NaN
f_final,1.0,NaN
abundance_reaction,0.0,NaN


In [10]:
def retrieve_variable_values_after_solution(com_model):
    variable_vals = {}
    for var in com_model.model.solver.variables:
        variable_vals[var.name] = var.primal
    return variable_vals

def is_close_to(val, target=0, t=10**-6):
    return abs(val - target) < t

def test_sum_of_gamma(var_vals):
    gamma_vars = {}
    for k, v in var_vals.items():
        target_string = "_fraction_reaction"
        if target_string in k[-len(target_string):]:
            gamma_vars[k] = v

    if is_close_to(sum(gamma_vars.values()), target=1.):
        print(f"Sum of gammas is 1!\n{gamma_vars}")
    else:
        print(f"{'!'*5} Sum of gammas is not 1!\n{gamma_vars}")

def test_equal_growth_rate(com_model, var_vals, target_growth_rate=None):
    community_gr = var_vals["community_biomass"]
    if target_growth_rate is not None:
        if is_close_to(community_gr, target=target_growth_rate):
            print(f"Community growth rate passed")
        else:
            print(f"{'!'*5} Community growth rate error: {community_gr} != {target_growth_rate}")
    else:
        target_growth_rate = community_gr
        print(f"Setting target to community growth rate {community_gr}")
    for name in com_model.get_member_names():
        bio_flux = var_vals[f"{name}_to_community_biomass"]
        gamma = var_vals[f"{name}_fraction_reaction"]
        if is_close_to(gamma):
            member_gr = None
            print(f"{name} growth not tested due to gamma being 0. Biomass flux is {bio_flux}")
            if not is_close_to(bio_flux):
                print(f"{'!'*5} Warning, biomass flux {bio_flux} despite no abundance {gamma}")
        else:
            member_gr = bio_flux / gamma
            if is_close_to(member_gr, target=target_growth_rate):
                print(f"{name} growth rate passed")
            else:
                print(f"{'!'*5} {name} growth rate error: {member_gr} != {target_growth_rate} at flux {bio_flux} and gamma {gamma}")
        

## Minimal and maximal growth rate ##
 - Minimal growth rate is never lower than mu_lb (test 0, 0.1, 2., 3.)
 - Maximum growth rate is never higher than mu_ub (test 1000., 2.1, 2., 1.9, 0.7, 0.)

In [11]:
for val in [0., 0.1, 2., 3.]:
    print(f"\nTesting mu_lb = {val}")
    com_model = create_toy_pycomo_model()
    pycomo.helper.discretized_growth.relax_model_linearisation(com_model)
    f_vars = pycomo.helper.discretized_growth.init_f_variables(com_model)
    pycomo.helper.discretized_growth.linearize_pycomo_model(com_model, f_vars, mu_lb=val, mu_ub=4., n_mu_bins=10)
    print(f"Testing BM_com")
    com_model.model.objective = "community_biomass"
    sol = com_model.model.optimize("minimize")
    if sol.status != "optimal":
        print(f"Solver status {sol.status}")
    print(f"Solver finished with {sol.status} at value {sol.objective_value}")
    var_vals = retrieve_variable_values_after_solution(com_model)
    test_sum_of_gamma(var_vals)
    test_equal_growth_rate(com_model, var_vals, target_growth_rate=val)

2026-03-15 15:39:38,132 - PyCoMo - INFO - No community model generated yet. Generating now:
2026-03-15 15:39:38,137 - PyCoMo - INFO - Identified biomass reaction from objective: bio
2026-03-15 15:39:38,137 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-03-15 15:39:38,138 - PyCoMo - INFO - Biomass reaction bio is a boundary reaction.
2026-03-15 15:39:38,138 - PyCoMo - INFO - Moving reactant BM out of the boundary compartment into compartment 'bio', to avoid further boundary reactions being added to it.
2026-03-15 15:39:38,160 - PyCoMo - INFO - Identified biomass reaction from objective: bio
2026-03-15 15:39:38,161 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-03-15 15:39:38,162 - PyCoMo - INFO - Biomass reaction bio is a boundary reaction.
2026-03-15 15:39:38,162 - PyCoMo - INFO - Moving reactant BM out of the boundary compartment into compartment 'bio', to avoid further boundary reactions being added


Testing mu_lb = 0.0


AttributeError: module 'pycomo.helper.discretized_growth' has no attribute 'linearize_pycomo_model'

Minimization works correctly. Solution was found at specified minimal growth rate - except for 3., where the problem was correctly identified as infeasible.

In [ ]:
for val in [1000., 2.1, 2., 1.9, 0.7, 0.]:
    print(f"\nTesting mu_ub = {val}")
    com_model = create_toy_pycomo_model()
    pycomo.helper.discretized_growth.relax_model_linearisation(com_model)
    f_vars = pycomo.helper.discretized_growth.init_f_variables(com_model)
    pycomo.helper.discretized_growth.linearize_pycomo_model(com_model, f_vars, mu_lb=0, mu_ub=val, n_mu_bins=4, constraint_limits=0.)
    print(f"Testing BM_com")
    com_model.model.objective = "community_biomass"
    sol = com_model.model.optimize("maximize")
    if sol.status != "optimal":
        print(f"Solver status {sol.status}")
    print(f"Solver finished with {sol.status} at value {sol.objective_value}")
    var_vals = retrieve_variable_values_after_solution(com_model)
    test_sum_of_gamma(var_vals)
    target_growth = 0.
    step_size = val / 4.
    for step in range(5):
        if step_size * step <= 2.:
            target_growth = step_size * step
        else:
            break
    print(f"Target growth rate at {target_growth} due to step_size {step_size}")
    test_equal_growth_rate(com_model, var_vals, target_growth_rate=target_growth)

The maximization of growth works as intended

## Bin size ##
 - With mu_lb = 0, mu_ub = 2 check bin num/size 4/0.5 10/0.2 1/2.0
 - With mu_lb = 0, mu_ub = 1000 check bin num/size 1000/1.0 100/10.
 - With mu_lb = 0, mu_ub = 4 check bin num/size 4/1.0 10/0.4 5/0.8
 - With mu_lb = 1., mu_ub = 3. check bin num/size 10/0.2 5/0.4
 - With mu_lb = 0.5, mu_ub = 2. check bin num/size 10/0.15 5/0.3

In [ ]:
bin_size_tests = {
    (0.,2.): [4,10,1],
    (0.,1000.): [1000,100],
    (0.,4.): [4,10,5],
    (1.,3.): [10,5],
    (0.5,2.): [10,5],
}

In [ ]:
for mu_bounds, bin_sizes in bin_size_tests.items():
    mu_lb, mu_ub = mu_bounds
    for val in bin_sizes:
        step_size = (mu_ub - mu_lb) / float(val)
        print(f"\nTesting bin size = {step_size} ({val} bins) with growth rate bounds {mu_lb}/{mu_ub}")
        com_model = create_toy_pycomo_model()
        pycomo.helper.discretized_growth.relax_model_linearisation(com_model)
        f_vars = pycomo.helper.discretized_growth.init_f_variables(com_model)
        pycomo.helper.discretized_growth.linearize_pycomo_model(com_model, f_vars, mu_lb=mu_lb, mu_ub=mu_ub, n_mu_bins=val)
        print(f"Testing BM_com")
        com_model.model.objective = "community_biomass"
        sol = com_model.model.optimize("maximize")
        if sol.status != "optimal":
            print(f"Solver status {sol.status}")
        print(f"Solver finished with {sol.status} at value {sol.objective_value}")
        var_vals = retrieve_variable_values_after_solution(com_model)
        test_sum_of_gamma(var_vals)
        target_growth = 0. if mu_lb > 2. else mu_lb
        for step in range(val+1):
            if mu_lb + step_size * step <= 2.:
                target_growth = mu_lb + step_size * step
            else:
                break
        print(f"Target growth rate at {target_growth} due to step_size {step_size}")
        test_equal_growth_rate(com_model, var_vals, target_growth_rate=target_growth)

### Now checking v4 ###

In [ ]:
for mu_bounds, bin_sizes in bin_size_tests.items():
    mu_lb, mu_ub = mu_bounds
    for val in bin_sizes:
        step_size = (mu_ub - mu_lb) / float(val)
        print(f"\nTesting bin size = {step_size} ({val} bins) with growth rate bounds {mu_lb}/{mu_ub}")
        com_model = create_toy_pycomo_model()
        pycomo.helper.discretized_growth.relax_model_linearisation(com_model)
        f_vars = pycomo.helper.discretized_growth.init_f_variables(com_model)
        pycomo.helper.discretized_growth.linearize_pycomo_model(com_model, f_vars, mu_lb=mu_lb, mu_ub=mu_ub, n_mu_bins=val)
        print(f"Testing BM_com")
        com_model.model.objective = "org1_v4"
        sol = com_model.model.optimize("maximize")
        if sol.status != "optimal":
            print(f"Solver status {sol.status}")
        print(f"Solver finished with {sol.status} at value {sol.objective_value}")
        var_vals = retrieve_variable_values_after_solution(com_model)
        test_sum_of_gamma(var_vals)
        test_equal_growth_rate(com_model, var_vals)

All tests are passed for v4. The optimum resides at a growth rate of 1 (or close to) and sits at one of the specified intervals given by the growth rate.

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=4., n_mu_bins=8)

In [ ]:
print(f"\nTesting BM_A_com")
com_model.model.objective = "org1_bio"
print(com_model.model.optimize().fluxes)
for var in com_model.model.solver.variables:
    print(var.name, var.primal)

In [ ]:
print(f"\nTesting BM_A_com")
com_model.model.objective = "org1_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_B_com")
com_model.model.objective = "org2_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_com")
com_model.model.objective = "community_biomass"
print(com_model.model.optimize().fluxes)
print(f"\nTesting TP_X1")
com_model.model.objective = "org1_TP_X1"
print(com_model.model.optimize().fluxes)
print(f"\nTesting v4A")
com_model.model.objective = "org1_v4"
print(com_model.model.optimize().fluxes)

### Test with mu_lb > 0. and mu_ub < 2. ###

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=.7, n_mu_bins=10)

print(f"\nTesting BM_A_com")
com_model.model.objective = "org1_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_B_com")
com_model.model.objective = "org2_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_com")
com_model.model.objective = "community_biomass"
print(com_model.model.optimize().fluxes)
print(f"\nTesting TP_X1")
com_model.model.objective = "org1_TP_X1"
print(com_model.model.optimize().fluxes)
print(f"\nTesting v4A")
com_model.model.objective = "org1_v4"
print(com_model.model.optimize().fluxes)

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=4., n_mu_bins=10)

print(f"\nTesting BM_A_com")
com_model.model.objective = "org1_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_B_com")
com_model.model.objective = "org2_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_com")
com_model.model.objective = "community_biomass"
print(com_model.model.optimize().fluxes)
print(f"\nTesting TP_X1")
com_model.model.objective = "org1_TP_X1"
print(com_model.model.optimize().fluxes)
print(f"\nTesting v4A")
com_model.model.objective = "org1_v4"
print(com_model.model.optimize().fluxes)

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=1.2, mu_ub=2., n_mu_bins=20)

print(f"\nTesting BM_A_com")
com_model.model.objective = "org1_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_B_com")
com_model.model.objective = "org2_bio"
print(com_model.model.optimize().fluxes)
print(f"\nTesting BM_com")
com_model.model.objective = "community_biomass"
print(com_model.model.optimize().fluxes)
print(f"\nTesting TP_X1")
com_model.model.objective = "org1_TP_X1"
print(com_model.model.optimize().fluxes)
for var in com_model.model.solver.variables:
    print(var.name, var.primal)
print(f"\nTesting v4A")
com_model.model.objective = "org1_v4"
print(com_model.model.optimize().fluxes)
for var in com_model.model.solver.variables:
    print(var.name, var.primal)

In [ ]:
raise AssertionError

In [ ]:
def get_mu_vars(com_model):
    mu_vars = []
    i = 0
    while True:
        mu_val = 2**i
        try:
            mu_var = com_model.model.solver.variables[str(mu_val)]
        except KeyError:
            break
        mu_vars.append(mu_var)
        i += 1
    return mu_vars

def add_binary_variables_to_all_rxns(com_model, no_f_rxns=True):
    activity_vars = []
    if no_f_rxns:
        target_reactions = list(set(com_model.model.reactions) - set(com_model.f_reactions))
    else:
        target_reactions = list(set(com_model.model.reactions))
    bigM = 10000.0
    for rxn in target_reactions:
        var_name = f"ao_{rxn.id}_abs"
        if var_name in com_model.model.variables:
            y_abs = com_model.model.variables[var_name]
        else:
            y_abs = com_model.model.solver.interface.Variable(var_name)
            com_model.model.solver.add([y_abs])
            c_abs1 = com_model.model.solver.interface.Constraint(y_abs - rxn.flux_expression, lb=0., name=f"{var_name}_lb")
            c_abs2 = com_model.model.solver.interface.Constraint(y_abs + rxn.flux_expression, lb=0., name=f"{var_name}_ub")
            com_model.model.solver.add([c_abs1, c_abs2])  
        
        var_name = f"ao_{rxn.id}_zero"
        if var_name in com_model.model.variables:
            y_zero = com_model.model.variables[var_name]
        else:
            y_zero = com_model.model.solver.interface.Variable(var_name, type="binary")
            com_model.model.solver.add([y_zero])

            v = rxn.flux_expression

            # v_i <= M (1 - b_i)
            c_zero1 = com_model.model.solver.interface.Constraint(v - bigM * (1 - y_zero), ub=0.)
        
            # -v_i <= M (1 - b_i)
            c_zero2 = com_model.model.solver.interface.Constraint(-v - bigM * (1 - y_zero), ub=0.)
            com_model.model.solver.add([c_zero1, c_zero2]) 

            # epsilon = 10**-2
            # c_zero3 = com_model.model.solver.interface.Constraint(rxn.forward_variable - epsilon * (1 - y_zero) + bigM * y_zero, lb=0.)
            # c_zero4 = com_model.model.solver.interface.Constraint(rxn.reverse_variable - epsilon * (1 - y_zero) + bigM * y_zero, lb=0.)
            # com_model.model.solver.add([c_zero3, c_zero4]) 
            #c_zero = com_model.model.solver.interface.Constraint(y_abs + bigM * y_zero, ub=bigM, lb=1., name=f"{var_name}_lb")
            #com_model.model.solver.add([c_zero])  
            
        # binary variable
        var_name = f"ao_{rxn.id}_fwd"
        if var_name in com_model.model.variables:
            y_fwd = com_model.model.variables[var_name]
        else:
            y_fwd = com_model.model.solver.interface.Variable(var_name, type="binary")
            com_model.model.solver.add([y_fwd])
            activity_vars.append(y_fwd)
            c_fwd1 = com_model.model.solver.interface.Constraint(rxn.flux_expression - bigM * y_fwd, ub=0, name=f"{var_name}_ub")
            c_fwd2 = com_model.model.solver.interface.Constraint(rxn.flux_expression - bigM * y_fwd, lb=-bigM, name=f"{var_name}_lb")
            c_fwd3 = com_model.model.solver.interface.Constraint(y_abs - y_fwd, lb=0., name=f"{var_name}_abs")
            com_model.model.solver.add([c_fwd1, c_fwd2, c_fwd3])
            
        # binary variable
        var_name = f"ao_{rxn.id}_rev"
        if var_name in com_model.model.variables:
            y_rev = com_model.model.variables[var_name]
        else:
            y_rev = com_model.model.solver.interface.Variable(var_name, type="binary")
            com_model.model.solver.add([y_rev])
            activity_vars.append(y_rev)
            c_rev1 = com_model.model.solver.interface.Constraint(rxn.flux_expression + bigM * y_rev, lb=0, name=f"{var_name}_lb")
            c_rev2 = com_model.model.solver.interface.Constraint(rxn.flux_expression + bigM * y_rev, ub=bigM, name=f"{var_name}_ub")
            c_rev3 = com_model.model.solver.interface.Constraint(y_abs - y_rev, lb=0., name=f"{var_name}_abs")
            com_model.model.solver.add([c_rev1, c_rev2, c_rev3])   

        c_total = com_model.model.solver.interface.Constraint(sum([y_zero, y_fwd, y_rev]), lb=1., ub=1., name=f"ao_{rxn.id}_total")
        com_model.model.solver.add([c_total])   
        
    return activity_vars


def add_binary_variables_to_all_rxns_via_indicators(com_model, eps=10**-6, no_f_rxns=True):
    activity_vars = []
    if no_f_rxns:
        target_reactions = list(set(com_model.model.reactions) - set(com_model.f_reactions))
    else:
        target_reactions = list(set(com_model.model.reactions))
    for rxn in target_reactions:
        
        var_name = f"ao_{rxn.id}_fwd"
        if var_name in com_model.model.variables:
            y_fwd = com_model.model.variables[var_name]
        else:
            y_fwd = com_model.model.solver.interface.Variable(var_name, type="binary")
            com_model.model.solver.add([y_fwd])
            activity_vars.append(y_fwd)
            c_fwd1 = com_model.model.solver.interface.Constraint(rxn.flux_expression, ub=0, name=f"{var_name}_ub", indicator_variable=y_fwd, active_when=0)
            c_fwd2 = com_model.model.solver.interface.Constraint(rxn.flux_expression, lb=eps, name=f"{var_name}_lb", indicator_variable=y_fwd, active_when=1)
            com_model.model.solver.add([c_fwd1, c_fwd2])
            
        # binary variable
        var_name = f"ao_{rxn.id}_rev"
        if var_name in com_model.model.variables:
            y_rev = com_model.model.variables[var_name]
        else:
            y_rev = com_model.model.solver.interface.Variable(var_name, type="binary")
            com_model.model.solver.add([y_rev])
            activity_vars.append(y_rev)
            c_rev1 = com_model.model.solver.interface.Constraint(rxn.flux_expression, lb=0, name=f"{var_name}_lb", indicator_variable=y_rev, active_when=0)
            c_rev2 = com_model.model.solver.interface.Constraint(rxn.flux_expression, ub=-eps, name=f"{var_name}_ub", indicator_variable=y_rev, active_when=1)
            com_model.model.solver.add([c_rev1, c_rev2])   

        c_total = com_model.model.solver.interface.Constraint(sum([y_fwd, y_rev]), ub=1., name=f"ao_{rxn.id}_total")
        com_model.model.solver.add([c_total])   
        
    return activity_vars


def eliminate_solution(com_model, solution, activity_vars, mu_vars=None, no_f_rxns=True):
    active_rxns = []

    for rxn_id, flux in solution.items():
        try:
            if flux > (10**-6):
                active_rxns.append(com_model.model.variables[f"ao_{rxn_id}_fwd"])
            elif flux < -(10**-6):
                active_rxns.append(com_model.model.variables[f"ao_{rxn_id}_rev"])
        except KeyError:
            continue

    inactive_rxns = list(set(activity_vars) - set(active_rxns))

    
    if mu_vars is not None:
        for mu_var in mu_vars:
            if mu_var.primal > (10**-6):
                active_rxns.append(mu_var)
            else:
                inactive_rxns.append(mu_var)
    
    n_active = len(active_rxns)

    print(f"Active: {len(active_rxns)}, inactive: {len(inactive_rxns)}")
    print(f"Active: {active_rxns}\n, inactive: {inactive_rxns}")

    free_cons_name = "ao_sol_1"
    i = 1

    while free_cons_name+"_active" in com_model.model.solver.constraints or free_cons_name+"_inactive" in com_model.model.solver.constraints:
        i += 1
        free_cons_name = f"ao_sol_{i}"
        
    print(f"Adding constraint {free_cons_name}")
    if active_rxns:
        c_elim_sol_active = com_model.model.solver.interface.Constraint(sum(active_rxns), ub=n_active-1, name=free_cons_name+"_active")
        com_model.model.solver.add([c_elim_sol_active])
    else:
        if no_f_rxns:
            target_reactions = list(set(com_model.model.reactions) - set(com_model.f_reactions))
        else:
            target_reactions = list(set(com_model.model.reactions))
        flux_vars = []
        for rxn in target_reactions:
            flux_vars.append(rxn.flux_expression)
        c_total = com_model.model.solver.interface.Constraint(sum(flux_vars), lb=10**-6, name=f"flux_total")
        com_model.model.solver.add([c_total])
        print(c_total)
    if inactive_rxns:
        c_elim_sol_inactive = com_model.model.solver.interface.Constraint(sum(inactive_rxns), lb=1., name=free_cons_name+"_inactive")
        com_model.model.solver.add([c_elim_sol_inactive])
    
    return

def find_alternative_optima(com_model, n=2, repeat=False, bin_vars=None, optima = None, print_variables=False, no_f_rxns=True, use_mu_vars=False):
    mu_vars = None
    if use_mu_vars:
        mu_vars = get_mu_vars(com_model)
        
    if optima is None:
        optima = {}
    if not repeat:
        bin_vars = add_binary_variables_to_all_rxns(com_model, no_f_rxns=no_f_rxns)
        print(bin_vars)
    
    while n > len(optima):
        i = len(optima) + 1
        optimum = com_model.model.optimize()
        print(f"optimum {i} found with objective value {optimum.objective_value} and status {optimum.status}")
        if optimum.status != "optimal":
            print(f"Aborting alternative optima search due to non optimal solution: {optimum.status}")
            break
        optima[i] = optimum
        if print_variables:
            for var in com_model.model.solver.variables:
                print(var.name, var.primal)
        try:
            eliminate_solution(com_model, solution=optimum.fluxes, activity_vars=bin_vars, no_f_rxns=no_f_rxns, mu_vars=mu_vars)
        except ValueError:
            print(f"Zero vector is optimum")
            break
    
    return optima
        
    

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=2., n_mu_bins=5)

In [ ]:
optima = {}
optima = find_alternative_optima(com_model, n=20, repeat=False, bin_vars=None, optima = optima, print_variables=True, use_mu_vars=True)

In [ ]:
optima[1].fluxes

In [ ]:
for idx, optimum in optima.items():
    print(f"Optimum {idx} at {optimum.objective_value} ({optimum.status}):")
    print(optimum.fluxes)

In [ ]:
optima[list(optima.keys())[-1]].fluxes

In [ ]:
optima[list(optima.keys())[-1]].status

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=2., n_mu_bins=10)
optima = {}
optima = find_alternative_optima(com_model, n=20, repeat=False, bin_vars=None, optima = optima, print_variables=True, no_f_rxns=False)

In [ ]:
for idx, optimum in optima.items():
    print(f"Optimum {idx} at {optimum.objective_value} ({optimum.status}):")
    print(optimum.fluxes)

In [ ]:
com_model = create_toy_pycomo_model()
com_model.save("toy_model_for_milp.xml")

In [ ]:
f_vars

In [ ]:
com_model = create_toy_pycomo_model()
relax_model_linearisation(com_model)
f_vars = init_f_variables(com_model)
linearize_pycomo_model(com_model, f_vars, mu_lb=0., mu_ub=2., n_mu_bins=10)
add_binary_variables_to_all_rxns_via_indicators(com_model, no_f_rxns=False)
com_model.model.reactions.community_biomass.bounds = (1.6,1.6)
model = com_model.model.solver
cplex_model = model.problem

In [ ]:
import cplex

In [ ]:
print(cplex_model.variables.get_types())

In [ ]:
print(cplex_model.variables.get_names())

In [ ]:
with open('model.lp', 'w') as out:
    out.write(str(com_model.model.solver))
cplex_model = cplex.Cplex("model.lp")

In [ ]:
cplex_model.parameters.mip.pool.intensity.set(4)
cplex_model.parameters.mip.pool.capacity.set(100)
cplex_model.parameters.mip.limits.populate.set(100)
cplex_model.populate_solution_pool()
num_solutions = cplex_model.solution.pool.get_num()

print(f"Number of solutions found: {num_solutions}")

for i in range(num_solutions):
    obj = cplex_model.solution.pool.get_objective_value(i)
    values = cplex_model.solution.pool.get_values(i)
    print(f"Solution {i}:\n{obj}\n\n{values}\n\n\n{'='*20}\n")

    for j in range(len(values)):
        print(f"{cplex_model.variables.get_names()[j]}: {values[j]}")


In [ ]:
num_solutions

In [ ]:
for i in range(num_solutions):
    obj = cplex_model.solution.pool.get_objective_value(i)
    values = cplex_model.solution.pool.get_values(i)
    print(f"Solution {i}:\n{obj}\n\n{values}\n\n\n{'='*20}\n")